# 🧭 Pipeline OE1 — Caracterización del Costo de Navegación ERP
## PPI_C9_2026 · Facultad de Ingeniería y Arquitectura · UPeU

---

**Título del proyecto:**  
*Desarrollo de un modelo local para la reestructuración de interfaces adaptativas orientado a la reducción del costo de navegación en sistemas ERP*

**Autores:** Gutierrez Anco Yesenia Alejandra · Quispe Huarilloclla Jhosef Antony · Turpo Cauna Jose Samuel

---

### 📋 Objetivo Específico 1 (OE1)
> *"Caracterizar el costo de navegación en sistemas ERP mediante la identificación y medición de métricas derivadas de los registros de interacción del usuario, tomando como referencia los modelos de análisis de patrones de acceso documentados en la literatura [Fu et al., Aung & Kumoi (2026).]"*  
> — PPI §1.5.2

### 🗺️ Estructura del pipeline

```
Stage 1 → Extracción       (leer logs JSON / PostgreSQL)
Stage 2 → Segmentación     (sessionización por timeout — Aung & Kumoi (2026). 2000)
Stage 3 → Filtración       (eliminar accesos < 3s y sesiones incompletas — PPI §2.3.1)
Stage 4 → Perfiles         (vectores de frecuencia normalizados — Aung & Kumoi (2026).)
Stage 5 → Métricas         (M1: clics · M2: tiempo · M3: tasa error — PPI §2.3.3)
Stage 6 → Exportación      (CSV + JSON listos para OE2 y OE5)
```

---


---
## ⚙️ Sección 0 · Configuración del entorno

Antes de correr cualquier celda, verificamos que todas las librerías necesarias estén instaladas.

| Librería | Uso en el pipeline |
|---|---|
| `pandas` | DataFrames para métricas y exportación |
| `numpy` | Cálculos estadísticos de costo |
| `pathlib` | Manejo de rutas de archivos |
| `json` | Lectura de logs en formato JSON |
| `hashlib` | Anonimización SHA-256 de IDs (ética §3.2.3) |
| `collections` | Conteo eficiente de frecuencias |


In [1]:
%pip install pandas numpy --quiet

import json
import hashlib
import logging
import warnings
from pathlib import Path
from datetime import datetime, timedelta
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from collections import defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

# Logger del pipeline
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
log = logging.getLogger("PPI-OE1")

print("✅ Entorno listo")
print(f"   pandas  {pd.__version__}")
print(f"   numpy   {np.__version__}")


Note: you may need to restart the kernel to use updated packages.
✅ Entorno listo
   pandas  2.2.2
   numpy   1.26.4


---
## ⚙️ Sección 1 · Configuración del pipeline (`PipelineConfig`)

Todos los parámetros del pipeline están centralizados en una sola clase.  
Esto facilita la reproducibilidad y permite correr el pipeline con distintas configuraciones  
sin tocar el código (buena práctica de investigación reproducible).

### Parámetros clave y su justificación metodológica

| Parámetro | Valor por defecto | Justificación |
|---|---|---|
| `session_timeout_min` | 30 min | Estándar en sessionización web (Aung & Kumoi (2026). 2000) |
| `min_stay_seconds` | 3 s | Umbral de acceso accidental definido en PPI §2.3.1 |
| `min_session_events` | 2 clics | Sesión con 1 clic → incompleta por corte de conexión |
| `anonymize` | `True` | Requerimiento ético PPI §3.2.3 |
| `menu_depth` | ver tabla | Profundidad real del árbol de menú del ERP Articulos |

### Profundidad del menú ERP Articulos

```
GRUPO (nivel 1)
  └── SUBMENU (nivel 2)
        └── ITEM (nivel 3)   ← destino navegable
```

Cada clic adicional de profundidad incrementa el **costo de navegación M1**.


In [2]:
@dataclass
class PipelineConfig:
    # ── Fuente de datos ──────────────────────────────────────────────
    input_path:          str  = "navigation_logs_augmented.ndjson"
    db_url:              str  = None  # Alternativa: URL de PostgreSQL

    # ── Segmentación de sesiones (Stage 2) ───────────────────────────
    session_timeout_min: int  = 30    # minutos sin actividad → nueva sesión

    # ── Filtración de ruido (Stage 3) ────────────────────────────────
    min_stay_seconds:    int  = 3     # accesos < 3s son "accidentales" (PPI §2.3.1)
    min_session_events:  int  = 2     # sesiones con 1 clic → incompletas

    # ── Privacidad (Stage 4) ─────────────────────────────────────────
    anonymize:           bool = True  # SHA-256 de user_id (ético §3.2.3)

    # ── Profundidad del árbol de menú del ERP Articulos ─────────────
    menu_depth: Dict = field(default_factory=lambda: {
        # Las rutas no listadas aquí asumen nivel 3 (peor caso)
        "configuracion/seguridad/users": 3,
        "configuracion/seguridad/roles": 3,
        "configuracion/seguridad/permisos": 3,
        "configuracion/seguridad/admin": 3,
        "configuracion/seguridad/cambio-clave": 3,
        "configuracion/puntos-venta": 2,
        "configuracion/puntos-atributos": 2,
        "configuracion/empresas": 2,
        "configuracion/sucursales": 2,
        "configuracion/catalogo-productos": 2,
        "configuracion/atributos-producto": 2,
        "configuracion/utilitarios": 2,
        "procesos/historial-movimientos": 2,
        "procesos/atributos-diverso": 2,
        "procesos/reimprimir-comprobantes": 2,
        "procesos/detalle-documentos": 2,
        "procesos/documento-paciente": 2,
        "procesos/transferencias": 2,
        "procesos/aprobaciones": 2,
        "reportes/correlatividad": 2,
        "reportes/compras": 2,
        "reportes/ventas": 2,
        "reportes/cajas": 2,
        "reportes/kardex": 2,
        "reportes/stock-valorizado": 2,
        "reportes/descuento": 2,
        "reportes/comprobantes-anulados": 2,
        "reportes/nota-credito": 2,
        "reportes/documentos-anulados": 2,
        "compra/orden": 2,
        "compra/registrar": 2,
        "compra/intercambio": 2,
        "compra/proveedores": 2,
        "compra/pagos-pendientes": 2,
        "compra/estado-cuenta": 2,
        "venta/listado": 2,
        "venta/cotizaciones": 2,
        "venta/notas-venta": 2,
        "venta/venta": 2,
        "venta/anulaciones": 2,
        "venta/notas-credito": 2,
        "venta/clientes": 2,
        "venta/devoluciones": 2,
        "venta/descuentos/lista": 3,
        "venta/descuentos/convenio": 3,
        "venta/comprobante-electronico": 2,
        "facturacion/resumenes": 2,
        "facturacion/anulaciones": 2,
        "caja/arqueos": 2,
        "caja/chica": 2,
        "caja/general": 2,
        "caja/medio-pago": 2,
        "caja/diferencia-costos": 2,
        "caja/cuentas": 2,
        "almacen/almacenes": 2,
        "almacen/productos": 2,
        "documentos": 2,
        "documentos/bloques": 2,
        "documentos/puntos": 2,
        "documentos/formatos": 2,
    })

    # ── Salida ────────────────────────────────────────────────────────
    output_dir: str = "output_oe1"


# Instanciar con valores por defecto (modificar aquí según necesidad)
cfg = PipelineConfig(
    input_path          = "navigation_logs_augmented.ndjson",
    session_timeout_min = 30,
    min_stay_seconds    = 3,
    anonymize           = False,   # False para desarrollo local
    output_dir          = "output_oe1"
)

print(f"Configuración cargada:")
print(f"  Archivo de entrada:   {cfg.input_path}")
print(f"  Timeout de sesión:    {cfg.session_timeout_min} minutos")
print(f"  Mínimo permanencia:   {cfg.min_stay_seconds} segundos")
print(f"  Anonimización:        {cfg.anonymize}")
print(f"  Directorio de salida: {cfg.output_dir}")


Configuración cargada:
  Archivo de entrada:   navigation_logs_augmented.ndjson
  Timeout de sesión:    30 minutos
  Mínimo permanencia:   3 segundos
  Anonimización:        False
  Directorio de salida: output_oe1


---
## 📦 Sección 2 · Modelos de datos

Definimos las estructuras que representan los datos a lo largo del pipeline.

### `RawEvent`
Representa **un solo clic de navegación** tal como llega del log.  
Campos mínimos según PPI §2.3.1:
> *"identificador de usuario, identificador de módulo/función accedida, timestamp de inicio y fin, y secuencia de navegación"*

### `NavigationSession`
Agrupa los eventos de un usuario en una **sesión de trabajo continua**.  
Una sesión termina cuando el usuario estuvo inactivo más de `session_timeout_min`.


In [3]:
@dataclass
class RawEvent:
    """Un clic de navegación en el ERP."""
    timestamp:   datetime
    user_id:     str
    role:        str
    action:      str    # "Read Kardex", "Create Orden Compra", etc.
    route:       str    # "almacen/kardex", "compra/orden", etc.
    exec_time_s: float  # tiempo de respuesta del servidor en segundos
    is_error:    bool   # si el servidor devolvió error HTTP


@dataclass
class NavigationSession:
    """Una sesión de trabajo de un usuario (conjunto de eventos consecutivos)."""
    session_id:  str
    user_id:     str
    role:        str
    start_time:  datetime
    end_time:    datetime
    events:      List[RawEvent] = field(default_factory=list)

    @property
    def duration_seconds(self) -> float:
        """Duración total de la sesión en segundos."""
        return (self.end_time - self.start_time).total_seconds()

    @property
    def n_clicks(self) -> int:
        """Número de clics (navegaciones) en la sesión."""
        return len(self.events)

print("✅ Modelos de datos definidos: RawEvent, NavigationSession")


✅ Modelos de datos definidos: RawEvent, NavigationSession


---
## 📥 Stage 1 · Extracción de datos

**Fuente:** Logs del ERP en formato JSON (generados por el módulo de captura Angular + Spring Boot)  
**Alternativa de producción:** Query directa a la tabla `navigation_logs` de PostgreSQL

### Formato esperado de cada registro

```json
{
  "timestamp":    "2026-05-01 08:14:32",
  "user_id":      "bot_Ventas_a3f2",
  "role":         "Ventas",
  "action":       "Read Registro Venta",
  "route":        "/api/venta/venta",
  "execution_time": 0.0823,
  "is_error":     false
}
```

### Dos modos de lectura

```
JSON   → para desarrollo y datos sintéticos (este notebook)
PostgreSQL → para producción (ERP real en producción)
```

La función `_parse_record()` normaliza cualquier variación en los campos  
(por ejemplo, `route` puede venir con o sin prefijo `/api/`).


In [4]:
class Stage1_Extract:
    """
    Stage 1: Extracción de logs desde JSON o PostgreSQL.
    
    CORRECCIÓN v2: manejo robusto de BOM UTF-8 (Windows),
    NDJSON, JSON array y archivos con encoding mixto.
    """
    def __init__(self, cfg: PipelineConfig):
        self.cfg = cfg

    def run(self) -> List[RawEvent]:
        log.info("── STAGE 1: Extracción ──────────────────────────────")
        if self.cfg.db_url:
            events = self._from_postgres()
        else:
            events = self._from_json()
        log.info(f"   Eventos cargados: {len(events):,}")
        return events

    def _from_json(self) -> List[RawEvent]:
        path = Path(self.cfg.input_path)
        if not path.exists():
            raise FileNotFoundError(
                f"\n❌ Archivo no encontrado: {path.resolve()}\n"
                f"   Asegúrate de que 'navigation_logs_augmented.ndjson' esté\n"
                f"   en la misma carpeta que este notebook."
            )

        # ── FIX WINDOWS: utf-8-sig elimina el BOM automáticamente ──
        raw = path.read_text(encoding="utf-8-sig").strip()

        records = []

        # Caso 1: JSON array  →  [ {...}, {...}, ... ]
        if raw.startswith("["):
            try:
                records = json.loads(raw)
                log.info("   Formato detectado: JSON array")
                return [self._parse_record(r) for r in records]
            except json.JSONDecodeError as e:
                raise ValueError(f"El archivo parece un JSON array pero no es válido: {e}")

        # Caso 2: NDJSON  →  una línea = un objeto JSON
        if raw.startswith("{"):
            errores = []
            for i, line in enumerate(raw.splitlines(), 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError as e:
                    errores.append(f"Línea {i}: {e}")

            if errores and not records:
                raise ValueError(
                    f"No se pudo parsear ninguna línea del archivo.\n"
                    f"Primeros errores: {errores[:3]}"
                )
            if errores:
                log.warning(f"   {len(errores)} líneas con error omitidas (de {i} total)")

            log.info("   Formato detectado: NDJSON (una línea = un objeto)")
            return [self._parse_record(r) for r in records]

        # Caso 3: formato desconocido → intentar json.loads directo
        try:
            data = json.loads(raw)
            if isinstance(data, list):
                log.info("   Formato detectado: JSON anidado")
                return [self._parse_record(r) for r in data]
            if isinstance(data, dict) and "data" in data:
                return [self._parse_record(r) for r in data["data"]]
        except json.JSONDecodeError:
            pass

        raise ValueError(
            f"❌ Formato de archivo no reconocido.\n"
            f"   El archivo debe ser JSON array [...] o NDJSON (un objeto por línea).\n"
            f"   Primeros 100 caracteres: {repr(raw[:100])}"
        )

    def _from_postgres(self) -> List[RawEvent]:
        from sqlalchemy import create_engine, text
        engine = create_engine(self.cfg.db_url)
        query = text("""
            SELECT
                to_char(timestamp, 'YYYY-MM-DD HH24:MI:SS') as timestamp,
                CONCAT('user_', role, '_', user_id::text)   as user_id,
                role,
                CONCAT('Read ', menu_item_titulo)            as action,
                '/' || route                                 as route,
                COALESCE(execution_time_ms / 1000.0, 0)     as execution_time,
                COALESCE(is_error, false)                    as is_error
            FROM navigation_logs
            ORDER BY timestamp ASC
        """)
        with engine.connect() as conn:
            rows = conn.execute(query).fetchall()
        return [self._parse_record(dict(r._mapping)) for r in rows]

    def _parse_record(self, r: dict) -> RawEvent:
        ts_str = r.get("timestamp", "")
        try:
            ts = datetime.strptime(ts_str, "%Y-%m-%d %H:%M:%S")
        except ValueError:
            try:
                ts = datetime.fromisoformat(ts_str)
            except ValueError:
                ts = datetime.now()
                log.warning(f"   Timestamp inválido: {ts_str!r}, usando now()")

        route = r.get("route", "").lstrip("/").replace("api/", "")

        return RawEvent(
            timestamp   = ts,
            user_id     = str(r.get("user_id", "unknown")),
            role        = str(r.get("role", "unknown")),
            action      = str(r.get("action", "")),
            route       = route,
            exec_time_s = float(r.get("execution_time", 0)),
            is_error    = bool(r.get("is_error", False)),
        )


# ── Ejecutar Stage 1 ────────────────────────────────────────────
raw_events = Stage1_Extract(cfg).run()

# Vista previa
df_preview = pd.DataFrame([{
    "timestamp":  e.timestamp,
    "user_id":    e.user_id,
    "role":       e.role,
    "route":      e.route,
    "exec_time_s":e.exec_time_s
} for e in raw_events[:5]])
print("\n📋 Primeros 5 eventos extraídos:")
df_preview

19:52:38 [INFO] ── STAGE 1: Extracción ──────────────────────────────
19:52:38 [INFO]    Formato detectado: NDJSON (una línea = un objeto)
19:52:38 [INFO]    Eventos cargados: 6,475



📋 Primeros 5 eventos extraídos:


,timestamp,user_id,role,route,exec_time_s
0,2026-05-01 07:00:35,bot_Logistica_7037,Logistica,reportes/kardex,0.1157
1,2026-05-01 07:01:46,bot_Logistica_7037,Logistica,almacen/productos,0.0685
2,2026-05-01 07:02:29,bot_Logistica_7037,Logistica,reportes/kardex,0.1867
3,2026-05-01 07:02:34,bot_Logistica_7037,Logistica,reportes/stock-valorizado,0.2046
4,2026-05-01 07:09:40,bot_Administrador_a86b,Administrador,configuracion/sucursales,0.0275


---
## 🔀 Stage 2 · Segmentación de sesiones (Sessionización)

**Referencia metodológica:**  
> *"Segmentación de sesiones: identificación de los límites de cada sesión de trabajo del usuario mediante heurísticas de timeout, análoga al procedimiento de «sessionización» descrito por Aung & Kumoi (2026)."* — PPI §2.3.1

### ¿Qué es una sesión?

Una **sesión** es un bloque de trabajo continuo de un usuario.  
Termina cuando el usuario estuvo **inactivo más de N minutos** (timeout).

```
Usuario A: [08:10 kardex] [08:12 stock] [08:45 productos]   ← timeout 30min → NUEVA SESIÓN
           [09:30 almacen] [09:32 kardex]
```

### Algoritmo

```
Para cada evento ordenado por (usuario, timestamp):
    Si el evento pertenece a un usuario diferente  →  nueva sesión
    Si pasaron más de 30 min desde el último clic →  nueva sesión
    Si no                                          →  agregar a sesión actual
```

El umbral de **30 minutos** es el estándar usado por Aung & Kumoi (2026). (2000)  
en su estudio de patrones de navegación sobre logs de msnbc.com.


In [5]:
class Stage2_Sessionize:
    """
    Stage 2: Agrupación de eventos en sesiones por timeout de inactividad.
    
    Referencia: Aung & Kumoi (2026) — timeout de 30 min estándar para análisis de clickstream
    Timeout estándar: 30 minutos (adoptado en PPI §2.3.1)
    """

    def __init__(self, cfg: PipelineConfig):
        self.timeout = timedelta(minutes=cfg.session_timeout_min)

    def run(self, events: List[RawEvent]) -> List[NavigationSession]:
        log.info("── STAGE 2: Segmentación de sesiones ────────────────")
        log.info(f"   Timeout: {self.timeout}")

        # Paso 1: ordenar por usuario y tiempo
        sorted_events = sorted(events, key=lambda e: (e.user_id, e.timestamp))

        sessions: List[NavigationSession] = []
        current: Optional[NavigationSession] = None

        for event in sorted_events:
            es_nuevo_usuario = current is None or event.user_id != current.user_id
            hay_timeout      = current is not None and                                (event.timestamp - current.events[-1].timestamp) > self.timeout

            if es_nuevo_usuario or hay_timeout:
                # Guardar sesión anterior si existe
                if current is not None:
                    sessions.append(current)

                # Crear nueva sesión
                sid = f"{event.user_id}_{event.timestamp.strftime('%Y%m%d%H%M%S')}"
                current = NavigationSession(
                    session_id = sid,
                    user_id    = event.user_id,
                    role       = event.role,
                    start_time = event.timestamp,
                    end_time   = event.timestamp,
                    events     = [event],
                )
            else:
                # Agregar evento a sesión actual
                current.events.append(event)
                current.end_time = event.timestamp

        if current:
            sessions.append(current)

        # Estadísticas
        clicks_por_sesion    = [s.n_clicks for s in sessions]
        duracion_por_sesion  = [s.duration_seconds / 60 for s in sessions]

        log.info(f"   Sesiones detectadas:    {len(sessions):,}")
        log.info(f"   Clics/sesión  — prom:   {np.mean(clicks_por_sesion):.1f}  "
                 f"min: {min(clicks_por_sesion)}  max: {max(clicks_por_sesion)}")
        log.info(f"   Duración/sesión — prom: {np.mean(duracion_por_sesion):.1f} min")

        return sessions


# ── Ejecutar Stage 2 ─────────────────────────────────────────────
sessions = Stage2_Sessionize(cfg).run(raw_events)

# Distribución de sesiones por usuario
df_ses = pd.DataFrame([{
    "usuario": s.user_id, "rol": s.role,
    "n_clics": s.n_clicks,
    "duracion_min": round(s.duration_seconds / 60, 1),
    "inicio": s.start_time.strftime("%Y-%m-%d %H:%M"),
} for s in sessions])

print("\n📊 Distribución de sesiones por rol:")
df_ses.groupby("rol").agg(
    sesiones=("usuario", "count"),
    clics_prom=("n_clics", "mean"),
    duracion_prom_min=("duracion_min", "mean")
).round(1)


19:52:42 [INFO] ── STAGE 2: Segmentación de sesiones ────────────────
19:52:42 [INFO]    Timeout: 0:30:00
19:52:42 [INFO]    Sesiones detectadas:    873
19:52:42 [INFO]    Clics/sesión  — prom:   7.4  min: 1  max: 51
19:52:42 [INFO]    Duración/sesión — prom: 26.2 min



📊 Distribución de sesiones por rol:


,sesiones,clics_prom,duracion_prom_min
rol,,,
Administrador,94,6.9000,20.2000
Caja,148,7.3000,27.5000
Compras,149,5.9000,19.9000
Logistica,160,8.8000,28.2000
Reportes,87,6.9000,23.8000
Ventas,235,7.9000,31.2000


---
## 🧹 Stage 3 · Filtración de ruido

**Referencia metodológica:**
> *"Filtración de ruido: eliminación de accesos accidentales (función accedida por error y abandonada en menos de tres segundos) y de sesiones incompletas por corte de conexión."* — PPI §2.3.1

### Dos tipos de ruido que eliminamos

**1. Accesos accidentales (< 3 segundos)**  
Si el usuario navegó a una función y salió en menos de 3 segundos,  
se considera que fue un clic por error. Conservar estos datos distorsionaría  
el perfil de navegación real del usuario.

```
[08:10:01 kardex]  [08:10:02 productos]  ← 1 segundo → accidental, ELIMINAR
[08:10:45 kardex]  [08:12:30 stock]      ← 105 segundos → uso real, CONSERVAR
```

**2. Sesiones incompletas**  
Sesiones con menos de 2 eventos son probablemente sesiones  
interrumpidas por corte de conexión o cierre accidental del navegador.

### ¿Por qué importa esta etapa?

Si no filtramos ruido, el modelo aprenderá patrones falsos:  
un usuario parecería "frecuentar" funciones que en realidad solo abrió por error.  
Esto degradaría la calidad de la reestructuración del menú.


In [6]:
class Stage3_Filter:
    """
    Stage 3: Filtración de accesos accidentales y sesiones incompletas.
    
    Criterios definidos en PPI §2.3.1:
    - Acceso accidental: tiempo hasta siguiente clic < 3 segundos
    - Sesión incompleta: menos de 2 eventos válidos
    """

    def __init__(self, cfg: PipelineConfig):
        self.min_stay   = cfg.min_stay_seconds    # 3 segundos por defecto
        self.min_events = cfg.min_session_events  # 2 eventos mínimo

    def run(self, sessions: List[NavigationSession]) -> List[NavigationSession]:
        log.info("── STAGE 3: Filtración de ruido ─────────────────────")

        total_sesiones_orig = len(sessions)
        total_eventos_orig  = sum(s.n_clicks for s in sessions)
        n_accidentales      = 0
        n_sesiones_cortas   = 0
        sesiones_limpias    = []

        for session in sessions:
            eventos_limpios = []

            for i, evt in enumerate(session.events):
                # Siempre descartar errores HTTP (is_error = True)
                if evt.is_error:
                    continue

                # Verificar tiempo de permanencia
                if i < len(session.events) - 1:
                    delta_seg = (session.events[i+1].timestamp - evt.timestamp).total_seconds()
                    if delta_seg < self.min_stay:
                        n_accidentales += 1
                        continue  # acceso accidental → descartar

                eventos_limpios.append(evt)

            # Actualizar sesión con eventos limpios
            session.events = eventos_limpios
            if eventos_limpios:
                session.start_time = eventos_limpios[0].timestamp
                session.end_time   = eventos_limpios[-1].timestamp

            # Descartar sesiones con muy pocos eventos
            if len(session.events) < self.min_events:
                n_sesiones_cortas += 1
                continue

            sesiones_limpias.append(session)

        total_eventos_limpio = sum(s.n_clicks for s in sesiones_limpias)

        log.info(f"   Sesiones:  {total_sesiones_orig:,} → {len(sesiones_limpias):,}  "
                 f"(-{n_sesiones_cortas} incompletas)")
        log.info(f"   Eventos:   {total_eventos_orig:,} → {total_eventos_limpio:,}  "
                 f"(-{n_accidentales} accidentales)")
        log.info(f"   Retención: {total_eventos_limpio/total_eventos_orig*100:.1f}% de los eventos")

        return sesiones_limpias


# ── Ejecutar Stage 3 ─────────────────────────────────────────────
clean_sessions = Stage3_Filter(cfg).run(sessions)

# Comparar antes vs después por rol
antes  = pd.DataFrame([{"rol": s.role, "clics": s.n_clicks} for s in sessions])
despues = pd.DataFrame([{"rol": s.role, "clics": s.n_clicks} for s in clean_sessions])

print("\n📊 Comparación antes / después del filtrado:")
comparacion = pd.DataFrame({
    "sesiones_antes":  antes.groupby("rol").size(),
    "sesiones_despues":despues.groupby("rol").size(),
    "clics_antes":     antes.groupby("rol")["clics"].sum(),
    "clics_despues":   despues.groupby("rol")["clics"].sum(),
})
comparacion["retencion_%"] = (comparacion["clics_despues"] / comparacion["clics_antes"] * 100).round(1)
comparacion


19:52:51 [INFO] ── STAGE 3: Filtración de ruido ─────────────────────
19:52:51 [INFO]    Sesiones:  873 → 761  (-112 incompletas)
19:52:51 [INFO]    Eventos:   6,475 → 6,180  (-31 accidentales)
19:52:51 [INFO]    Retención: 95.4% de los eventos



📊 Comparación antes / después del filtrado:


,sesiones_antes,sesiones_despues,clics_antes,clics_despues,retencion_%
rol,,,,,
Administrador,94,80,631,617,97.8000
Caja,148,128,1052,1032,98.1000
Compras,149,127,856,834,97.4000
Logistica,160,147,1393,1380,99.1000
Reportes,87,78,582,573,98.5000
Ventas,235,201,1775,1744,98.3000


---
## 👤 Stage 4 · Vectores de perfil de usuario

**Referencia metodológica:**  
> *"Para cada usuario, se generará un vector de frecuencia de acceso por función, normalizado por el número total de sesiones, siguiendo el esquema de vectores de características propuesto por Aung & Kumoi (2026)."* — PPI §2.3.1

### ¿Qué es un vector de perfil?

Para cada usuario construimos un vector donde cada dimensión es una ruta del ERP  
y el valor es cuántas veces por sesión (en promedio) accedió a esa ruta:

```
usuario_A = {
    "almacen/kardex":      1.8,   ← accede 1.8 veces por sesión en promedio
    "almacen/stock":       1.2,
    "compra/orden":       0.3,
    "venta/venta":        0.0,   ← nunca accede (no es su rol)
    ...
}
```

### También construimos la matriz de transiciones

Para cada par de rutas consecutivas que navegó el usuario:
```
kardex → stock-valorizado   (transición)
stock  → productos           (transición)
```

Estas transiciones alimentarán las **GRU/Transformer** del OE2.

### Privacidad (PPI §3.2.3)

Con `anonymize=True`, el `user_id` se reemplaza por su hash SHA-256 truncado:
```
"bot_Ventas_a3f2" → "u_9c4f7a1b"
```
Este proceso es **irreversible** — cumple el requerimiento ético del PPI.


In [7]:
class Stage4_ProfileVectors:
    """
    Stage 4: Construcción de vectores de perfil de usuario.

    Extrae:
    - Vector de frecuencia normalizada por usuario (v_u) → alimenta score compuesto OE2
    - Secuencias de rutas por sesión → alimenta GRU/Transformer en OE2
    - Frecuencia por rol → alimenta prior de rol en OE2
    """

    def __init__(self, cfg: PipelineConfig):
        self.anonymize = cfg.anonymize

    def run(self, sessions: List[NavigationSession]) -> Dict:
        log.info("── STAGE 4: Vectores de perfil de usuario ───────────")

        user_sessions: Dict[str, List[NavigationSession]] = defaultdict(list)
        for s in sessions:
            uid = self._hash_uid(s.user_id) if self.anonymize else s.user_id
            user_sessions[uid].append(s)

        all_routes = sorted({
            e.route
            for s in sessions
            for e in s.events
            if e.route
        })

        profiles = {}

        for uid, u_sessions in user_sessions.items():
            freq_abs: Dict[str, int] = defaultdict(int)
            # Secuencias de rutas por sesión (input para GRU/Transformer en OE2)
            secuencias: List[List[str]] = []
            role = u_sessions[0].role

            for sess in u_sessions:
                rutas = [e.route for e in sess.events if e.route]
                for r in rutas:
                    freq_abs[r] += 1
                if len(rutas) >= 2:
                    secuencias.append(rutas)

            n_sesiones = len(u_sessions)
            freq_norm  = {r: cnt / n_sesiones for r, cnt in freq_abs.items()}
            top5       = sorted(freq_abs.items(), key=lambda x: x[1], reverse=True)[:5]

            profiles[uid] = {
                "user_id":       uid,
                "role":          role,
                "n_sesiones":    n_sesiones,
                "freq_absoluta": dict(freq_abs),
                "freq_norm":     freq_norm,
                "secuencias":    secuencias,   # ← OE2: input para GRU/Transformer
                "top5_rutas":    top5,
                "all_routes":    all_routes,
            }

        log.info(f"   Perfiles generados:      {len(profiles):,} usuarios")
        log.info(f"   Rutas únicas en corpus:  {len(all_routes):,}")
        total_seq = sum(len(p["secuencias"]) for p in profiles.values())
        log.info(f"   Secuencias para GRU/TF:  {total_seq:,}")
        return profiles

    def _hash_uid(self, user_id: str) -> str:
        """Anonimización SHA-256 irreversible (PPI §3.2.3)."""
        return "u_" + hashlib.sha256(user_id.encode()).hexdigest()[:8]


# ── Ejecutar Stage 4 ─────────────────────────────────────────────
profiles = Stage4_ProfileVectors(cfg).run(clean_sessions)

sample_uid = list(profiles.keys())[1]
p = profiles[sample_uid]

print(f"\n👤 Perfil de ejemplo: {sample_uid} (rol: {p['role']})")
print(f"   Sesiones totales:              {p['n_sesiones']}")
print(f"   Secuencias capturadas (≥2):   {len(p['secuencias'])}")
print(f"\n   Top 5 rutas más visitadas:")
for ruta, freq in p["top5_rutas"]:
    barra = "█" * int(freq / max(f for _, f in p["top5_rutas"]) * 20)
    print(f"   {ruta:35s} {barra} ({freq}x)")
print(f"\n   Ejemplo de secuencia: {p['secuencias'][0] if p['secuencias'] else []}")


19:53:17 [INFO] ── STAGE 4: Vectores de perfil de usuario ───────────
19:53:17 [INFO]    Perfiles generados:      28 usuarios
19:53:17 [INFO]    Rutas únicas en corpus:  44
19:53:17 [INFO]    Secuencias para GRU/TF:  761



👤 Perfil de ejemplo: bot_Caja_1125 (rol: Caja)
   Sesiones totales:              32
   Secuencias capturadas (≥2):   32

   Top 5 rutas más visitadas:
   caja/general                        ████████████████████ (109x)
   caja/arqueos                        ██████ (36x)
   venta/notas-credito                 █████ (32x)
   reportes/cajas                      █████ (29x)
   caja/cuentas                        ████ (26x)

   Ejemplo de secuencia: ['caja/arqueos', 'reportes/cajas', 'caja/cuentas', 'caja/chica', 'caja/cuentas', 'caja/general', 'caja/general', 'caja/chica', 'caja/general', 'caja/general', 'caja/arqueos', 'caja/general', 'caja/general', 'venta/notas-credito', 'caja/arqueos', 'reportes/cajas', 'caja/cuentas', 'caja/cuentas', 'caja/chica']


---
## 📏 Stage 5 · Métricas de costo de navegación

**Referencia metodológica:**
> *"La variable dependiente principal es el costo de navegación, operacionalizado a través de tres indicadores"* — PPI §2.3.3

### Las 3 métricas del PPI

| Métrica | Fórmula | Interpretación |
|---|---|---|
| **M1** Clics desde raíz | `profundidad_en_árbol_de_menú` | Cuántos clics necesita el usuario para llegar a la función (1=GRUPO, 2=SUBMENU, 3=ITEM) |
| **M2** Tiempo hasta función objetivo | `timestamp_top1 - timestamp_inicio_sesión` | Cuántos segundos tarda en llegar a su función más frecuente |
| **M3** Tasa de error de navegación | `clics_no_óptimos / total_clics` | Proporción de clics que no llevan al destino deseado |

### ¿Por qué estas tres métricas?

- **M1** captura el **esfuerzo estructural** — cuán profundo está enterrado el ítem en el menú
- **M2** captura el **tiempo real** — cuánto del tiempo de trabajo se pierde navegando
- **M3** captura la **desorientación** — cuántas veces el usuario se equivoca de camino

Estas métricas forman la **línea de base PRE-adaptación**.  
Después de implementar el modelo (OE3), las mismas métricas se miden de nuevo  
y la diferencia demuestra la efectividad del sistema (protocolo OE5).


In [8]:
class Stage5_NavigationCost:
    """
    Stage 5: Cálculo de las 3 métricas de costo de navegación (PPI §2.3.3).

    M1 — Clics promedio para llegar a función objetivo desde la raíz del menú
    M2 — Tiempo (seg) desde inicio de sesión hasta primera función objetivo
    M3 — Tasa de error: proporción de clics fuera de la trayectoria óptima
    """

    def __init__(self, cfg: PipelineConfig):
        self.menu_depth = cfg.menu_depth

    def run(self, sessions: List[NavigationSession], profiles: Dict) -> pd.DataFrame:
        log.info("── STAGE 5: Métricas de costo de navegación ─────────")
        rows = []

        for session in sessions:
            if not session.events:
                continue

            uid = session.user_id

            # ── M1: Clics promedio desde raíz ─────────────────────────
            clics = [self._profundidad(e.route) for e in session.events]
            m1 = np.mean(clics) if clics else 0

            # ── M2: Tiempo hasta función top1 del perfil ──────────────
            # FIX: verificar que top5_rutas no esté vacío antes de acceder [0]
            perfil    = profiles.get(uid, {})
            top5      = perfil.get("top5_rutas", [])
            top1_ruta = top5[0][0] if top5 else ""
            m2        = self._tiempo_hasta_ruta(session, top1_ruta)

            # ── M3: Tasa de error de navegación ───────────────────────
            m3 = self._tasa_error(session)

            rows.append({
                "session_id":         session.session_id,
                "user_id":            uid,
                "role":               session.role,
                "fecha":              session.start_time.date(),
                "hora_inicio":        session.start_time.hour,
                "n_clics":            session.n_clicks,
                "duracion_min":       round(session.duration_seconds / 60, 2),
                "M1_costo_clics":     round(m1, 3),
                "M2_tiempo_top1_s":   round(m2, 1),
                "M3_tasa_error":      round(m3, 4),
                "costo_sesion_total": round(m1 * session.n_clicks, 2),
            })

        df = pd.DataFrame(rows)

        if not df.empty:
            log.info(f"   Sesiones procesadas:    {len(df):,}")
            log.info(f"   M1 costo clics prom:    {df['M1_costo_clics'].mean():.3f}")
            log.info(f"   M2 tiempo top1 prom:    {df['M2_tiempo_top1_s'].mean():.1f} s")
            log.info(f"   M3 tasa error prom:     {df['M3_tasa_error'].mean():.4f}")

        return df

    # ── Funciones auxiliares ────────────────────────────────────────

    def _profundidad(self, route: str) -> int:
        """
        Devuelve la profundidad del ítem en el árbol de menú.
        1 = GRUPO (1 clic), 2 = SUBMENU (2 clics), 3 = ITEM (3 clics).
        Si la ruta no está mapeada, asume 3 (peor caso conservador).
        """
        return self.menu_depth.get(route, 3)

    def _tiempo_hasta_ruta(self, session: NavigationSession, target: str) -> float:
        """
        Segundos desde el inicio de sesión hasta el primer acceso a 'target'.
        Si target es vacío o el usuario nunca llegó, retorna la duración total.
        """
        if not target:
            return session.duration_seconds
        for evt in session.events:
            if evt.route == target:
                return (evt.timestamp - session.start_time).total_seconds()
        return session.duration_seconds

    def _tasa_error(self, session: NavigationSession) -> float:
        """
        Tasa de clics que no avanzan hacia un destino nuevo.
        Proxy: clics consecutivos sobre la misma ruta → usuario confundido.
        """
        if session.n_clicks <= 1:
            return 0.0
        errores = sum(
            1 for i in range(1, len(session.events))
            if session.events[i].route == session.events[i - 1].route
        )
        return errores / session.n_clicks


# ── Ejecutar Stage 5 ─────────────────────────────────────────────
df_costs = Stage5_NavigationCost(cfg).run(clean_sessions, profiles)

print("\n📊 Métricas de costo de navegación — LÍNEA DE BASE (pre-adaptación):")
resumen_roles = df_costs.groupby("role").agg(
    sesiones      = ("session_id",      "count"),
    M1_clics_prom = ("M1_costo_clics",   "mean"),
    M2_tiempo_s   = ("M2_tiempo_top1_s", "mean"),
    M3_error_pct  = ("M3_tasa_error",    lambda x: x.mean() * 100),
).round(2)
resumen_roles.columns = ["Sesiones", "M1 Clics (prom)", "M2 Tiempo s (prom)", "M3 Error %"]
resumen_roles

19:53:26 [INFO] ── STAGE 5: Métricas de costo de navegación ─────────
19:53:26 [INFO]    Sesiones procesadas:    761
19:53:26 [INFO]    M1 costo clics prom:    2.078
19:53:26 [INFO]    M2 tiempo top1 prom:    419.8 s
19:53:26 [INFO]    M3 tasa error prom:     0.0762



📊 Métricas de costo de navegación — LÍNEA DE BASE (pre-adaptación):


,Sesiones,M1 Clics (prom),M2 Tiempo s (prom),M3 Error %
role,,,,
Administrador,80,2.7000,454.9900,3.5900
Caja,128,2.0000,324.3400,14.0400
Compras,127,2.0000,384.9000,4.2500
Logistica,147,2.0000,311.7300,6.8900
Reportes,78,2.0000,423.9600,3.8600
Ventas,201,2.0200,566.1700,9.2600


---
## 💾 Stage 6 · Exportación de resultados

Esta etapa genera todos los archivos de salida del OE1,  
cada uno pensado como entrada para una etapa posterior del PPI.

| Archivo | Contenido | Usado en |
|---|---|---|
| `costo_navegacion_baseline.csv` | M1, M2, M3 por sesión | **OE5** evaluación pre/post |
| `perfiles_usuario.json` | Vectores de frecuencia normalizados | **OE2** modelo neuronal (GRU/Transformer) |
| `vocabulario_rutas.json` | Pares origen → destino | **OE2** GRU/Transformer |
| `frecuencia_por_rol.csv` | Accesos por ruta agrupados por rol | **OE3** algoritmo de reestructuración |
| `resumen_oe1.json` | Estadísticas del corpus + métricas | **Informe PPI** |

### Trazabilidad del pipeline

```
raw_navigation_logs.json
        ↓ Stage 1
    RawEvents (3,645)
        ↓ Stage 2
    Sessions (516)
        ↓ Stage 3
    CleanSessions (404)
        ↓ Stage 4
    Profiles (11 usuarios)
        ↓ Stage 5
    df_costs (404 filas × 10 columnas)
        ↓ Stage 6
    output_oe1/ [5 archivos]
```


In [9]:
class Stage6_Export:
    """
    Stage 6: Exportación de artefactos del OE1.
    Produce exactamente los archivos que OE2 (GRU/Transformer/DQN) necesita.
    """

    def __init__(self, cfg: PipelineConfig):
        self.out = Path(cfg.output_dir)
        self.out.mkdir(parents=True, exist_ok=True)

    def run(self, df_costs: pd.DataFrame, profiles: Dict,
            sessions: List[NavigationSession]) -> dict:
        log.info("── STAGE 6: Exportación ─────────────────────────────")

        # 1. Métricas baseline → CSV (OE5)
        p1 = self.out / "costo_navegacion_baseline.csv"
        df_costs.to_csv(p1, index=False, encoding="utf-8")
        log.info(f"   ✓ {p1.name}  ({len(df_costs)} filas)")

        # 2. Perfiles de usuario → JSON (OE2: frecuencia + secuencias para GRU/TF)
        export_perfiles = [{
            "user_id":    p["user_id"],
            "role":       p["role"],
            "n_sesiones": p["n_sesiones"],
            "freq_norm":  p["freq_norm"],
            "secuencias": p["secuencias"],   # ← secuencias de rutas para GRU/Transformer
            "top5_rutas": [{"ruta": r, "freq": f} for r, f in p["top5_rutas"]],
        } for p in profiles.values()]
        p2 = self.out / "perfiles_usuario.json"
        p2.write_text(json.dumps(export_perfiles, indent=2, ensure_ascii=False))
        log.info(f"   ✓ {p2.name}  ({len(export_perfiles)} perfiles)")

        # 3. Vocabulario de rutas → JSON (OE2: encoding para embeddings)
        all_routes = sorted({
            e.route for s in sessions for e in s.events if e.route
        })
        vocab = {"rutas": all_routes, "vocab_size": len(all_routes)}
        p3 = self.out / "vocabulario_rutas.json"
        p3.write_text(json.dumps(vocab, indent=2, ensure_ascii=False))
        log.info(f"   ✓ {p3.name}  ({len(all_routes)} rutas únicas)")

        # 4. Frecuencia por rol → JSON (OE2: prior de rol para score compuesto)
        rol_freq = defaultdict(lambda: defaultdict(int))
        for s in sessions:
            for e in s.events:
                if e.route:
                    rol_freq[s.role][e.route] += 1
        # Normalizar por total de accesos del rol
        rol_freq_norm = {}
        for rol, rutas in rol_freq.items():
            total = sum(rutas.values())
            rol_freq_norm[rol] = {r: round(cnt/total, 6) for r, cnt in rutas.items()}
        p4 = self.out / "frecuencia_por_rol.json"
        p4.write_text(json.dumps(rol_freq_norm, indent=2, ensure_ascii=False))
        # También CSV para compatibilidad OE3
        df_rol = pd.DataFrame([
            {"role": rol, "ruta": ruta, "accesos": cnt}
            for rol, rutas in rol_freq.items()
            for ruta, cnt in sorted(rutas.items(), key=lambda x: -x[1])
        ])
        p4b = self.out / "frecuencia_por_rol.csv"
        df_rol.to_csv(p4b, index=False, encoding="utf-8")
        log.info(f"   ✓ {p4.name} + {p4b.name}")

        # 5. Resumen → JSON
        resumen = {
            "oe1_completado": True,
            "fecha_ejecucion": datetime.now().isoformat(),
            "corpus": {
                "total_eventos":  sum(s.n_clicks for s in sessions),
                "total_sesiones": len(sessions),
                "total_usuarios": len(profiles),
                "roles": list({s.role for s in sessions}),
                "periodo": {
                    "inicio": str(min(s.start_time.date() for s in sessions)),
                    "fin":    str(max(s.end_time.date()   for s in sessions)),
                },
            },
            "metricas_baseline": {
                "M1_clics_prom":   round(df_costs["M1_costo_clics"].mean(),   3),
                "M2_tiempo_s_prom":round(df_costs["M2_tiempo_top1_s"].mean(), 3),
                "M3_error_prom":   round(df_costs["M3_tasa_error"].mean(),    4),
                "por_rol": df_costs.groupby("role").agg(
                    M1=("M1_costo_clics",   "mean"),
                    M2=("M2_tiempo_top1_s", "mean"),
                    M3=("M3_tasa_error",    "mean"),
                ).round(3).to_dict(orient="index"),
            },
        }
        p5 = self.out / "resumen_oe1.json"
        p5.write_text(json.dumps(resumen, indent=2, ensure_ascii=False, default=str))
        log.info(f"   ✓ {p5.name}")
        log.info(f"\n   Directorio: {self.out.resolve()}/")
        return resumen


# ── Ejecutar Stage 6 ─────────────────────────────────────────────
resumen = Stage6_Export(cfg).run(df_costs, profiles, clean_sessions)
print("\n✅ Exportación completa")


19:53:32 [INFO] ── STAGE 6: Exportación ─────────────────────────────
19:53:32 [INFO]    ✓ costo_navegacion_baseline.csv  (761 filas)
19:53:32 [INFO]    ✓ perfiles_usuario.json  (28 perfiles)
19:53:32 [INFO]    ✓ vocabulario_rutas.json  (44 rutas únicas)
19:53:32 [INFO]    ✓ frecuencia_por_rol.json + frecuencia_por_rol.csv
19:53:32 [INFO]    ✓ resumen_oe1.json
19:53:32 [INFO] 
   Directorio: D:\investigacion\Laboratorio\output_oe1/



✅ Exportación completa


---
## ✅ Sección Final · Resumen del OE1 y conexión con OE2–OE5

Esta celda consolida los resultados del OE1 y muestra  
cómo cada salida conecta con los siguientes objetivos del PPI.


In [10]:
# ═══════════════════════════════════════════════════════════
# RESUMEN EJECUTIVO OE1
# ═══════════════════════════════════════════════════════════

m = resumen["metricas_baseline"]
c = resumen["corpus"]

print("═" * 62)
print("  OE1 COMPLETADO — Caracterización del Costo de Navegación")
print("═" * 62)
print(f"\n  CORPUS")
print(f"  {'Período analizado:':25s} {c['periodo']['inicio']} → {c['periodo']['fin']}")
print(f"  {'Eventos procesados:':25s} {c['total_eventos']:,}")
print(f"  {'Sesiones válidas:':25s} {c['total_sesiones']:,}")
print(f"  {'Usuarios perfilados:':25s} {c['total_usuarios']}")
print(f"  {'Roles detectados:':25s} {', '.join(c['roles'])}")

print(f"\n  MÉTRICAS BASELINE (pre-adaptación)")
print(f"  {'M1 Costo clics prom:':25s} {m['M1_clics_prom']} clics/ítem")
print(f"  {'M2 Tiempo top1 prom:':25s} {m['M2_tiempo_s_prom']} segundos")
print(f"  {'M3 Tasa error prom:':25s} {m['M3_error_prom']*100:.2f}%")

print(f"\n  ARCHIVOS GENERADOS → output_oe1/")
archivos = {
    "costo_navegacion_baseline.csv": "→ OE5  Evaluación pre/post adaptación",
    "perfiles_usuario.json":         "→ OE2  Frecuencias + secuencias para GRU/Transformer",
    "vocabulario_rutas.json":        "→ OE2  Vocabulario de embeddings (VOCAB_SIZE)",
    "frecuencia_por_rol.json":       "→ OE2  Prior de rol para score compuesto",
    "frecuencia_por_rol.csv":        "→ OE3  Algoritmo de reestructuración de menú",
    "resumen_oe1.json":              "→ OE4  Documentación y benchmarking",
}
for archivo, destino in archivos.items():
    print(f"  ✓ {archivo:40s} {destino}")

print("\n" + "═" * 62)
print("  PRÓXIMO PASO → OE2: GRU / Transformer + Agente DQN")
print("  Entradas: perfiles_usuario.json + vocabulario_rutas.json")
print("            + frecuencia_por_rol.json")
print("═" * 62)


══════════════════════════════════════════════════════════════
  OE1 COMPLETADO — Caracterización del Costo de Navegación
══════════════════════════════════════════════════════════════

  CORPUS
  Período analizado:        2026-05-01 → 2026-05-28
  Eventos procesados:       6,180
  Sesiones válidas:         761
  Usuarios perfilados:      28
  Roles detectados:         Ventas, Administrador, Compras, Reportes, Caja, Logistica

  MÉTRICAS BASELINE (pre-adaptación)
  M1 Costo clics prom:      2.078 clics/ítem
  M2 Tiempo top1 prom:      419.829 segundos
  M3 Tasa error prom:       7.62%

  ARCHIVOS GENERADOS → output_oe1/
  ✓ costo_navegacion_baseline.csv            → OE5  Evaluación pre/post adaptación
  ✓ perfiles_usuario.json                    → OE2  Frecuencias + secuencias para GRU/Transformer
  ✓ vocabulario_rutas.json                   → OE2  Vocabulario de embeddings (VOCAB_SIZE)
  ✓ frecuencia_por_rol.json                  → OE2  Prior de rol para score compuesto
  ✓ frecuencia